In [16]:
import networkx as nx
import random
import time
from collections import deque

# --- Configuration for Reproducibility ---

random.seed(42)

def load_realistic_subgraph(filename, target_nodes=2000):
    """
    Loads the road network, subsamples a connected component using BFS,
    and assigns degree-based capacities.
    """
    print(f"Loading {filename} and creating subsampled network...")
    G = nx.Graph()
    try:
        with open(filename, 'r') as f:
            for line in f:
                if line.startswith('#'): continue
                parts = line.strip().split()
                if len(parts) >= 2:
                    u, v = int(parts[0]), int(parts[1])
                    G.add_edge(u, v)
    except FileNotFoundError:
        print("Error: roadNet-CA.txt not found. Cannot proceed.")
        return None

    # Subsample: BFS from a high-degree node
    degrees_full = dict(G.degree())
    start_node = max(degrees_full.items(), key=lambda x: x[1])[0]
    nodes_to_keep = set([start_node])
    queue = deque([start_node])
    
    while len(nodes_to_keep) < target_nodes and queue:
        curr = queue.popleft()
        for n in G.neighbors(curr):
            if n not in nodes_to_keep:
                nodes_to_keep.add(n)
                queue.append(n)
                if len(nodes_to_keep) >= target_nodes: break
    
    G_sub = G.subgraph(nodes_to_keep).copy()
    DG = nx.DiGraph()
    degrees_sub = dict(G_sub.degree())

    # Degree-based capacity modeling
    for u, v in G_sub.edges():
        cap = 50 + 10 * (degrees_sub.get(u, 0) + degrees_sub.get(v, 0))
        # Add edges in both directions
        DG.add_edge(u, v, capacity=cap, flow=0)
        DG.add_edge(v, u, capacity=cap, flow=0)
        
    print(f"Subsampled Model: {DG.number_of_nodes()} nodes, {DG.number_of_edges()} directed edges")
    return DG

def bfs_augmenting_path(residual_graph, s, t, parent_map):
    """
    Finds the shortest augmenting path from s to t using BFS.
    """
    
    visited = {node: False for node in residual_graph.nodes()}
    queue = deque()
    queue.append(s)
    visited[s] = True
    
    while queue:
        u = queue.popleft()
        for v in residual_graph.neighbors(u):
            if not visited[v] and residual_graph[u][v]['capacity'] > 0:
                queue.append(v)
                visited[v] = True
                parent_map[v] = u
                if v == t:
                    return True 
    return False

def get_reachable_nodes(residual_graph, s):
    """
    Performs BFS on the final residual graph to find all nodes 
    reachable from the source 's'. These nodes form the 'S' set of the Min Cut.
    """
    reachable = set()
    queue = deque([s])
    reachable.add(s)
    
    while queue:
        u = queue.popleft()
        for v in residual_graph.neighbors(u):
            # A node is reachable if the residual capacity is > 0
            if v not in reachable and residual_graph[u][v]['capacity'] > 0:
                reachable.add(v)
                queue.append(v)
                
    return reachable

def edmonds_karp(G, s, t):
    """
    Calculates the maximum flow and identifies the minimum cut.
    """
    R = G.copy() 
    max_flow = 0
    start_time = time.time()
    parent_map = {node: None for node in R.nodes()}
    
    # 1. Edmonds-Karp Algorithm (Max Flow)
    while bfs_augmenting_path(R, s, t, parent_map):
        path_flow = float('inf')
        v = t
        while v != s:
            u = parent_map[v]
            path_flow = min(path_flow, R[u][v]['capacity'])
            v = u
            
        max_flow += path_flow
        
        # Update residual capacities
        v = t
        while v != s:
            u = parent_map[v]
            R[u][v]['capacity'] -= path_flow
            if not R.has_edge(v, u):
                R.add_edge(v, u, capacity=0, flow=0)
            R[v][u]['capacity'] += path_flow
            v = u
            
    end_time = time.time()
    
    # 2. Minimum Cut Identification
    # Set S: Nodes reachable from s in the final residual graph
    S_set = get_reachable_nodes(R, s)
    T_set = set(R.nodes()) - S_set
    
    min_cut_edges = []
    cut_capacity_check = 0
    
    # Min Cut edges are all edges (u, v) in the ORIGINAL graph G where u is in S and v is in T
    for u, v in G.edges():
        if u in S_set and v in T_set:
            min_cut_edges.append((u, v))
            cut_capacity_check += G[u][v]['capacity']
    
    # --- Results Output ---

    print(f"Source (s): {s}, Sink (t): {t}")
    print(f"Maximum Flow: {max_flow:.1f} vehicles/hr")
    print(f"Computation Time: {end_time - start_time:.4f} seconds")

    print("\n--- Minimum Cut Analysis (The Bottleneck) ---")
    
    # The minimum cut partition
    print(f"Min Cut Partition Size (S-Set): {len(S_set)} nodes")
    
    # The Min Cut Edges (Bottleneck Roads)
    print(f"Number of Bottleneck Edges (Min Cut): {len(min_cut_edges)}")
    print("Bottleneck Edges (Origin Node ID, Destination Node ID, Capacity):")
    
    # Display the top 10 bottleneck edges and their capacities
    bottleneck_details = []
    for u, v in min_cut_edges:
        bottleneck_details.append((u, v, G[u][v]['capacity']))

    # Sort bottlenecks by capacity to prioritize major roads
    bottleneck_details.sort(key=lambda x: x[2], reverse=True)
    
    # Display the most critical bottleneck edges
    for u, v, cap in bottleneck_details[:10]:
        print(f"  ({u}, {v}): Capacity {cap:.1f} VPH")
        
    # Verification of Max Flow Min Cut Theorem
    print(f"\nMax Flow Min Cut Theorem Check:")
    print(f"Max Flow Value: {max_flow:.1f}")
    print(f"Min Cut Capacity (Sum of Bottleneck Edge Capacities): {cut_capacity_check:.1f}")
    
    # The slight difference between Max Flow and Min Cut Capacity is due to
    # the undirected nature of the original graph and how reverse edges are handled,
    # but the theory holds. The bottleneck list shows the physical road segments.
    
    return max_flow

# --- Execution ---

if _name_ == "_main_":

    # Load and prepare the graph
    flow_graph = load_realistic_subgraph('roadNet-CA.txt', target_nodes=2000)

    if flow_graph:
        nodes = list(flow_graph.nodes())
        
        # Select source and sink using the seeded random function (Reproducible)
        source, sink = random.sample(nodes, 2)
        
        # Run the Edmonds-Karp algorithm
        max_flow_result = edmonds_karp(flow_graph, source, sink)

Loading roadNet-CA.txt and creating subsampled network...
Subsampled Model: 10000 nodes, 31942 directed edges
Source (s): 658780, Sink (t): 559117
Maximum Flow: 330.0 vehicles/hr
Computation Time: 0.0200 seconds

--- Minimum Cut Analysis (The Bottleneck) ---
Min Cut Partition Size (S-Set): 1 nodes
Number of Bottleneck Edges (Min Cut): 3
Bottleneck Edges (Origin Node ID, Destination Node ID, Capacity):
  (658780, 658776): Capacity 110.0 VPH
  (658780, 658778): Capacity 110.0 VPH
  (658780, 658961): Capacity 110.0 VPH

Max Flow Min Cut Theorem Check:
Max Flow Value: 330.0
Min Cut Capacity (Sum of Bottleneck Edge Capacities): 330.0
